In [ ]:
import git
from os import listdir as ls
import numpy as np
import pandas as pd
import json
import yaml as yml
import pickle
from datetime import datetime
import pycountry
import pycountry_convert as pc
import random
from numpyro import distributions as dist
import arviz as az
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
from IPython.display import display, Markdown, Latex

from emu_renewal.selection import get_mob_avail_countries, gather_who_data, find_absent_inds, \
    find_neg_inds, find_outliers, find_nans_repeats
from emu_renewal.constants import DATA_PATH, FULL_RUN, TIMEOUTS, RERUNS, WANING_COMPARISON, WANING_TIMEOUTS, WANING_RERUNS, \
    MOB_LOCATION_SOURCE_MAP, MOB_LOCATION_NAME_MAP, CONT_CMAP, BASE_PATH, ANALYSIS_NAMES
from emu_renewal.geospatial import FacebookMobilityBuilder
from emu_renewal.inputs import get_worldbank_national_pop, get_income_group, get_fb_visited_mobility, \
    get_fb_singletile_mobility, process_raw_google_mobility, get_google_mobility, get_country_pop, \
    get_undesa_national_pop, get_country_vacc_data, get_world_shp
from emu_renewal.renew import MultiStrainModel
from emu_renewal.process import _get_cos_curve_at_x
from emu_renewal.mobility import NoMobilityProvider, WeightedExpMobilityProvider, SingleSeriesExpMobilityProvider
from emu_renewal.run import find_run_start_time, find_run_end_time, get_hosp_target, get_seroprev_target, \
    get_mobility_provider, run_calibration
from emu_renewal.indicators import get_deaths_target, get_cases_target, filter_seroprev, get_owid_hosps, \
    get_all_seroprev, get_seroprev_pooled_totals, get_var_target, extract_specific_var, get_country_vars, \
    get_alpha_info, get_delta_info, get_specific_var_props, \
    get_ba2_info, get_ba5_info
from emu_renewal.outputs import load_targets, get_country_procs, get_param_vals_by_analysis, get_idatas_for_mob_type, \
    get_ratios_from_disps, get_median_ratios, get_quantmedian_df, convert_quant_df_for_display, get_table_df_from_priors_dict, \
    add_bool_row_to_table, add_mob_avail_to_world
from emu_renewal.priors import get_standard_priors
from emu_renewal.distributions import GammaDens
from emu_renewal.calibration import custom_init, StandardCalib
from emu_renewal.plotting import plot_multianalysis_fit, add_cont_to_world_geodf, plot_continent_grouping, plot_prior_multipost, \
    compare_proc_mob, compare_proc_weighted_gmob, plot_proc_comparison, plot_kde_comparison, plot_mob_exp_violins, \
    plot_waning_comparison_proc_disp, plot_waning_quant_comparison, plot_input_recovery, plot_mob_weights_by_country, \
    plot_beta_priors, plot_duration_params, plot_inclusion
from emu_renewal.utils import get_countries_by_continent, get_analysis_paths, get_analysis_commits_df, split_list_into_segments, \
    get_country_short_name
from emu_renewal.document import get_func_blurb

plt.style.use("default")
set_matplotlib_formats("svg")

{{< pagebreak >}}

# Documentation

In [ ]:
repo = git.Repo(search_parent_directories=True)
commit_id = repo.git.rev_parse("--short", "HEAD")
txt = (
    "The following document was documented algorithmically "
    "from the code used in the analysis. "
)
txt += f"This version was produced from commit {commit_id}. \n\n"
txt += (
    "The rationale for our methods is provided in the "
    "Methods section of the main manuscript.\n\n"
)
txt += "# Analysis methods \n\n"
display(Markdown(txt))

## Summary equation
The mechanics of the renewal process can be summarised in the following equations, the detail of which is expanded in the following sections.
$$i_{p,v,t}=n_{p,t}x_{p,v}(1-e^{-\lambda_{v,t}})$$
$$\lambda_{v,t}=\beta\sigma_{v}e^{W_{t}}M_{t}\sum_{\tau=t-50}^{t-1}{(s_{v,\tau}+\sum_{p'}i_{p',v,\tau})g_{t-\tau}}$$
$$n_{p,t}=n_{p,t-1}-\sum_{v}i_{p,v,t}+i_{\tilde{p},\tilde{v},t}-qzn_{p,t-1}+qzr_{p}\sum_{p'}{n_{p',t-1}}$$
$$M_{t}=
\begin{cases}
1 & \text{if no mobility} \\
(\frac{\sum_{l}w_{l}G_{l,t}}{\sum_{l}w_{l}})^{m} & \text{if Google mobility} \\
F_{t}^{m} & \text{if Facebook mobility} \\
\end{cases}
$$
$$o_{t}=a\sum_{v}(\prod_{v'=v_{1}}^{v}{y_{v'}})\sum_{p}\sum_{\tau=1}^{50}i_{p,v,t-\tau} \times c_{\tau}$$
$$h_{t}=\sum_{\tau=1}^{50}o_{t-\tau}(1-\sum_{u=0}^{\tau-1}d_{u})$$
Where:

- $i_{p,v,t}$ represents the incidence of variant $v$ in immunity status group $p$ at time $t$, where $p$ represents the immunity status prior to infection with variant $v$
- $v$ takes sequential values from $v_1$ to $v_V$, where $V$ is the number of variants included in the analysis
- $p\in \{0, 1\}^{V}$ are the possible combination of past variant exposure statuses with $V$ variants
- $n_{p,t}$ is the size of the population with immunity status $p$ at time $t$
- $q$ indicates whether waning immunity is incorporated (i.e. $q=0$ base case analysis, $q=1$ waning immunity sensitivity analysis)
- $z$ is the reciprocal of the duration of natural immunity
- $r_{p}$ indicates whether $p=\{0\}^V$ and represents the infection-naive population (i.e. $r=1$ if $p=\{0\}^V$ and $r=0$ otherwise)
- $x_{p,v}$ is the relative susceptibility of persons with immunity status $p$ to infection with variant $v$
- $x_{\{0\}^{V},v} = 1$, full susceptibility for those never previously infected
- $x_{p,v} = 0$ for infection with a variant to which the person has previously been infected with or a preceding variant (e.g. infection with Alpha following infection with Delta)
- $x_{p,v}$ for immunity statuses other than the two preceding cases is the complement of a single calibrated cross-immunity parameter with a beta-distributed prior
- $\beta$ is the infectiousness of the starting variant
- $\sigma_{v}$ is the infectiousness of variant $v$ relative to the starting variant
- $W_{t}$ is the non-mechanistic residual transmission process, under which a Wiener sequence of updates from a starting value of zero is linked with a cosine spline at time $t$
- $s_{v,t}$ is the rate of seeding with variant $v$ at time $t$
- $g_{t}$ is $\int_{t}^{t+1} g(u)\,du$, where $g$ is the probability density function of the generation interval distribution
- $\tilde{v}$ is the variant for which infection results in transition from immunity status $\tilde{p}$ to immunity status $p$
- $G_{l,t}$ is the Google mobility estimate at location $l$ and time $t$
- $F_{t}$ is the Facebook mobility estimate (either tiles visited or single tile) at time $t$
- $w_{l}$ is a calibrated location-specific weight parameter with support $[0, 1]$
- $o_{t}$ is any one of the incident indicator quantities calculated from the incidence time series, i.e. either deaths, case notifications, hospital admission or ICU admissions
- $y_{v}$ is the relative severity of the infecting variant relative to the preceding variant, with $y_{1}=1$ (i.e. the severity of the starting strain is set to one)
- $c_{t}$ is $\int_{t}^{t+1} c(u)\,du$, where $c$ represents the probability density function of the distribution of the delay from the onset of the infection epiosde to the onset of the incident indicator considered
- $a$ is the proportion of infection episodes resulting in the indicator considered
- $h_{t}$ is either one of the prevalent indicators calculated from the incident indicators, i.e. either hospital occupancy or ICU occupancy
- $d_{t}$ is $\int_{t}^{t+1} d(u)\,du$, where $d$ represents the probability density function of the distribution of the time from the onset of the incident indicator (either hospital or ICU admission) to the end of the prevalent indicator period (either hospital or ICU discharge)

In [ ]:
txt = "\n\n## Renewal model\n"
model_rationale = (
    "We developed a discrete renewal model "
    "with daily innovations. The core aspect of our "
    "approach was the convolution of the preceding "
    "strain-specific incidence rate with "
    "the generation interval for subsequent infections, "
    "adjusted for past immunity. "
    "We then applied convolutions to this model "
    "to estimate model outputs that could be compared "
    "against our indicator targets described below.\n\n"
)
txt += model_rationale
dummy_start = datetime(2020, 1, 1)
dummy_end = datetime(2021, 12, 31)
no_mob_provider = NoMobilityProvider()
dummy_model = MultiStrainModel(1.0, dummy_start, dummy_end, [""], dummy_start, no_mob_provider, False, False)
txt += get_func_blurb(dummy_model.renew)

txt += "\n\n### Outputs\n"
txt += get_func_blurb(dummy_model.renewal_func) + "\n\n"
dummy_dist = GammaDens()
txt += get_func_blurb(dummy_dist.get_params) + "\n\n"

txt += "## Modelled population size \n\n"
txt += get_func_blurb(get_country_pop)
txt += get_func_blurb(get_worldbank_national_pop)
txt += get_func_blurb(get_undesa_national_pop)
txt += "\n\n## Analysis period\n"
analysis_period_rationale = (
    "For all countries, we aimed to place our analysis period "
    "during a time period for which the variation in transmission rate "
    "may have been substantially attributable to changes in mobility, "
    "although we developed an approach (described below) to address "
    "the consideration that the emergence of new SARS-CoV-2 variants "
    "likely contributed substantially. "
    "Therefore, we aimed to select time periods during which "
    "the roll-out of vaccination programs would have contributed "
    "less to variations in transmission rate.\n\n"
    "For all countries other than Oceania and Singapore, we addressed this "
    "by limiting our analysis period to the time prior to "
    "vaccination reaching levels that may have had a significant "
    "population-level effect.\n\n"
    "For most countries, a single base case no mobility analysis was run, "
    "against which the Google and Facebook analyses could be compared. "
    "However, for Oceania and Singapore, the analysis period for the Facebook "
    "analysis was shorter due to the discontinuation of Facebook "
    "data after May 2022. For these countries, an additional "
    "no mobility comparison analysis was run. "
)
txt += analysis_period_rationale
txt += get_func_blurb(find_run_start_time) + "\n\n"
txt += get_func_blurb(find_run_end_time)
txt += get_func_blurb(get_country_vacc_data)
txt += "\n\n## Mobility\n"
mobility_rationale = (
    "The main purpose of our analysis was to understand "
    "the effect of empirically observed changes in mobility "
    "on the observed fluctuations of the COVID-19 epidemic. "
    "We addressed this by including various mobility observation "
    "streams available from Big Tech companies within our model.\n\n"
)
txt += mobility_rationale
txt += get_func_blurb(get_mobility_provider)
txt += "\n\n### No mobility\n"
txt += get_func_blurb(NoMobilityProvider.__init__)
txt += "\n\n### Google mobility\n"
n_domains = 6
dummy_g_mob = pd.DataFrame(1.0, index=range(10), columns=range(n_domains))
dummy_g_priors = {"mob_weights": dist.Uniform(np.zeros(n_domains), np.ones(n_domains)), "mob_exp": None}
weighted_exp_mob_provider = WeightedExpMobilityProvider(dummy_g_mob, dummy_g_priors)
txt += get_func_blurb(process_raw_google_mobility)
txt += get_func_blurb(get_google_mobility)
txt += get_func_blurb(weighted_exp_mob_provider.__init__)
txt += get_func_blurb(weighted_exp_mob_provider.get_parameterised_mobility)
txt += "\n\n### Facebook mobility\n"
dummy_fb_mob_builder = FacebookMobilityBuilder()
txt += get_func_blurb(dummy_fb_mob_builder.build_mobility) + "\n\n"
txt += get_func_blurb(get_fb_visited_mobility)
txt += get_func_blurb(get_fb_singletile_mobility) + "\n\n"
dummy_fb_mob = pd.Series(1.0, index=range(10))
dummy_fb_priors = {"mob_exp": None}
single_series_mob_provider = SingleSeriesExpMobilityProvider(dummy_fb_mob, dummy_fb_priors)
txt += get_func_blurb(single_series_mob_provider.get_parameterised_mobility)
txt += get_func_blurb(single_series_mob_provider.__init__)
txt += "\n\n## Residual transmission process\n"
process_rationale = (
    "We included a non-mechanistic residual process "
    "within our model, which was intended to capture "
    "any variations in transmission that could not be "
    "attributed to the explicitly modelled processes "
    "(variant strain emergence, changes in mobility "
    "and the development of population immunity).\n\n"
)
txt += get_func_blurb(dummy_model.fit_process_curve)
txt += get_func_blurb(_get_cos_curve_at_x)
txt += get_func_blurb(dummy_model.initialise_var_proc)
txt += "\n\n## Calibration targets\n"
indicator_rationale = (
    "We only included countries for which multiple "
    "data streams were available to estimate variations "
    "in transmission rates over time. "
    "Specifically, we required that a time series for both "
    "COVID-19-attributable deaths and case notifications were available. "
    "However, we also included one time series for a hospital-related "
    "indicator, where this could be sourced. "
    "Because we considered that the emergence of new "
    "SARS-CoV-2 variants could have substantially affected transmission, "
    "we included explicit calibration targets for the "
    "strain-specific proportional prevalence of variants where available. "
    "Although possibly less epidemiologically relevant to "
    "this analysis, we sought to broadly capture the "
    "overall size of the epidemic. This was addressed by "
    "explicitly incorporating susceptible depletion and "
    "including seroprevalence estimates as calibration targets where available. "
    "We did not include vaccination explicitly within the model "
    "because our approach to country and time period inclusion "
    "was intended to focus on time periods during which "
    "vaccination would not have substantially influenced transmission rates.\n\n"
)
txt += indicator_rationale
txt += "\n\n### WHO indicators\n"
txt += get_func_blurb(get_deaths_target)
txt += get_func_blurb(get_cases_target)
txt += "\n\n### Hospitalisations\n"
txt += get_func_blurb(get_owid_hosps)
txt += get_func_blurb(get_hosp_target) + "\n\n"
txt += "\n\n### Seroprevalence\n"
txt += get_func_blurb(get_all_seroprev)
txt += get_func_blurb(filter_seroprev)
txt += get_func_blurb(get_seroprev_pooled_totals) + "\n\n"
txt += get_func_blurb(get_seroprev_target) + "\n\n"
txt += get_func_blurb(get_income_group)
txt += "\n\n### Variants\n\n"
variant_rationale = (
    "We included up to three key variants explicitly within "
    "our analysis framework. For all countries "
    "other than Oceania and Singapore "
    "our approach was intended to represent strains prior to the "
    "emergence of the Alpha variant and strains of Alpha "
    "with strains of Delta also included where sufficient data "
    "were available and the time period simulated meant that "
    "the emergence of Delta was relevant to the simulation. "
    "For Oceania and Singapore, we included pre-BA.2 strains of Omicron, "
    "along with Omicron BA.2 and Omicron BA.5. \n\n"
)
txt += "#### Data extraction\n"
txt += get_func_blurb(get_country_vars)
txt += get_func_blurb(extract_specific_var)
txt += get_func_blurb(get_specific_var_props) + "\n\n"
txt += get_func_blurb(get_var_target)
txt += "\n\n#### Alpha\n"
txt += get_func_blurb(get_alpha_info)
txt += "\n\n#### Delta\n"
txt += get_func_blurb(get_delta_info)
txt += "\n\n#### Omicron BA.2\n"
txt += get_func_blurb(get_ba2_info)
txt += "\n\n#### Omicron BA.5\n"
txt += get_func_blurb(get_ba5_info)
display(Markdown(txt))

In [ ]:
txt = "\n\n## Calibration algorithm\n"
calibration_rationale = (
    "We used a gradient-based algorithm "
    "for model calibration, which efficiently "
    "explored our multidimensional parameter space. "
    "By exploring all time series in log-space "
    "with a common dispersion parameter, "
    "we were able to apply an algorithm that "
    "provided epidemiologically plausible results for all "
    "countries simulated without the need for "
    "extensive country-specific adaptations.\n\n"
)
txt += calibration_rationale
txt += "\n\n### Initialisation\n"
txt += get_func_blurb(custom_init)
txt += "\n\n### Parameter processing\n"
dummy_calib = StandardCalib(dummy_model, {}, {})
txt += get_func_blurb(dummy_calib.sample_calib_params)
txt += get_func_blurb(dummy_calib.__init__)
txt += "\n\n"
txt += get_func_blurb(get_standard_priors)
txt += "\n\n### Algorithm\n"
txt += get_func_blurb(run_calibration)
display(Markdown(txt))

In [ ]:
txt = "# Selection of countries for inclusion\n"
either_mob_avail, summary, g_avail, fb_avail = get_mob_avail_countries()
txt += get_func_blurb(get_mob_avail_countries)
death_data, case_data = gather_who_data(either_mob_avail)
txt += get_func_blurb(gather_who_data) + "\n\n"
no_deaths, no_cases = find_absent_inds(death_data, case_data, summary)
txt += get_func_blurb(find_absent_inds)
neg_deaths, neg_cases = find_neg_inds(death_data, case_data, summary)
txt += get_func_blurb(find_neg_inds)
death_outliers, case_outliers = find_outliers(death_data, case_data, summary)
death_nans, case_nans, death_repeats, case_repeats = find_nans_repeats(death_data, case_data, summary)
txt += get_func_blurb(find_nans_repeats) + "\n\n"
txt += "The final results of the selection are presented in @tbl-inc. "
txt += "The final included countries are illustrated in @fig-inc below. "
display(Markdown(txt))

In [ ]:
excluded = set(no_deaths + no_cases + neg_deaths + neg_cases + death_nans + case_nans + death_repeats + case_repeats + death_outliers + case_outliers)
included = [c for c in either_mob_avail if c not in excluded]
add_bool_row_to_table(summary, included, "Incl-uded")

In [ ]:
#| label: fig-inc
#| fig-cap: Included countries and mobility availability. Countries coloured according to mobility availability. Neither source available, grey; Google mobility only available, green; Facebook mobility only available, blue; both sources available, purple. Red markers indicate included in analysis.
world = get_world_shp()
world["geometry"] = world.simplify(tolerance=0.1)
add_mob_avail_to_world(world, g_avail, fb_avail)
world["included"] = world["ISO_A3"].isin(included)
plot_inclusion(world);

In [ ]:
#| label: tbl-inc
#| tbl-cap: Reasons for country inclusion and exclusion.
summary.index = summary.index.map(lambda iso3: pycountry.countries.lookup(iso3).name)
display(Markdown(summary.sort_index().to_markdown()))

\newpage
# Parameter choices and supporting evidence

In [ ]:
loaded_priors = yml.safe_load(open(BASE_PATH / "data/evidence/priors.yml", "r"))
duration_params = loaded_priors["durations"]

## Time period parameters

In [ ]:
col_widths = '{tbl-colwidths="[12, 7, 7, 74]"}'
durations_df = get_table_df_from_priors_dict(loaded_priors["durations"])
keep_cols = [c for c in durations_df if c != "Short_name"]
dur_table = durations_df[keep_cols]
caption = ": Parameters and supporting evidence to time period priors. "
Markdown(dur_table.to_markdown() + "\n" + caption + col_widths)

In [ ]:
#| fig-cap: Duration-related parameter prior distributions.
plot_duration_params(duration_params)

{{< pagebreak >}}

## Proportion parameters

In [ ]:
betas_df = get_table_df_from_priors_dict(loaded_priors["beta"])
caption = ": Parameters and supporting evidence to beta-distributed priors. "
Markdown(betas_df.to_markdown() + "\n" + caption + col_widths)

In [ ]:
#| fig-cap: Proportion parameter prior distributions. Note horizontal axis range differs between upper two panels and lower three panels.
plot_beta_priors(loaded_priors["beta"])

In [ ]:
fixed_params = loaded_priors["fixed"]
params_df = pd.DataFrame.from_dict(fixed_params).T
params_df = params_df.set_index("param_name")
params_df.columns = params_df.columns.str.capitalize()
params_df.index.name = None
caption = "\n: Fixed parameters and supporting evidence. "

{{< pagebreak >}}

## Fixed parameters

In [ ]:
Markdown(params_df.to_markdown() + caption + '{tbl-colwidths="[12, 7, 81]"}')

{{< pagebreak >}}

## Between-variant relative parameters

In [ ]:
Markdown(loaded_priors["other"]["relinfect"]["evidence"])
Markdown(loaded_priors["other"]["relseverity"]["evidence"])

\newpage
# Calibration results
Results of individual model calibrations to epidemiological targets 
(with and without waning immunity included) were too large for inclusion
in this document and so are presented on Figshare
[here](https://bridges.monash.edu/articles/report/Comparison_of_model_outputs_to_calibration_targets_for_Evaluating_the_effects_of_population_mobility_on_transmission_during_the_COVID-19_pandemic_/32338440).
This section presents the comparisons of prior and posterior
distributions for each individual country analysis.

## Country grouping
Throughout our extended results,
we group countries according to the following continent groupings.

In [ ]:
#| fig-align: center
world = get_world_shp()
add_cont_to_world_geodf(world)
plot_continent_grouping(world)

\newpage

In [ ]:
#| fig-align: center
plt.style.use("ggplot")
all_countries = json.load(open(DATA_PATH / "config/included.json", "r"))
paths = {
    "base case": get_analysis_paths(RERUNS + FULL_RUN + TIMEOUTS, all_countries),
    "waning immunity included": get_analysis_paths(WANING_RERUNS + WANING_COMPARISON + WANING_TIMEOUTS, all_countries),
}
countries_by_cont = get_countries_by_continent(all_countries)
for include_waning in paths:
    txt = f"## Prior posterior comparisons by continent and country, {include_waning}\n" \
        "This section presents comparisons of parameter prior distributions to " \
        "their corresponding posterior distributions."
    display(Markdown(txt))
    display(Latex(r"\newpage"))
    for co, (cont, countries) in enumerate(countries_by_cont.items()):
        if co:
            display(Latex(r"\newpage"))
        display(Markdown(f"\n### {pc.convert_continent_code_to_continent_name(cont)}"))
        for c, country in enumerate(countries):
            country_name = pycountry.countries.lookup(country).name
            if c:
                display(Latex(r"\newpage"))
            display(Markdown(f"#### {country_name}"))
            analyses = paths[include_waning][country]

            no_mob_path = analyses["no_mob"]
            targs = load_targets(no_mob_path)
                
            priors = pickle.load(open(no_mob_path / "priors.pkl", "rb"))
            idatas = {a: az.from_netcdf(p / "idata_filtered.nc") for a, p in analyses.items()}
            var_names = ["start"] + [v[5:] for v in targs.keys() if v.startswith("prop_")]
            display(plot_prior_multipost(var_names, priors, idatas, 4))


# Comparison of the non-mechanistic scaling of transmission process with mobility domains
This section presents the residual transmission scaling (with uncertainty) implemented without scaling for mobility for each country. It is intended to provide a raw comparison between the residual non-mechanistic transmission that needed to be applied for each analysis in the absence of mobility scaling and the various mobility data fields available from both Google and Facebook. We also show comparisons against the composite Google mobility metric after weighting, with associated credible intervals.
From this section onward, results are presented for the model configuration without waning immunity included.

In [ ]:
analysis_paths = paths["base case"]
notes = {
    "g_mob": "Mobility values presented as Google data divided by 100, plus one.",
    "fb_visited_mob": "Mobility values presented as one plus Facebook data.",
    "fb_singletile_mob": "Mobility values presented as one minus Facebook data.",
}
display(Markdown("\\newpage ## Individual mobility metric comparisons (Google and Facebook)\n\n"))
for m, mob_type in enumerate(MOB_LOCATION_SOURCE_MAP):
    mob_source = MOB_LOCATION_SOURCE_MAP[mob_type]
    mob_name = MOB_LOCATION_NAME_MAP[mob_type]
    if m:
        display(Latex(r"\newpage"))
    section_title = f"### {mob_name} mobility metric comparison\n\n"
    display(Markdown(section_title))
    display(Markdown(notes[mob_source]))
    mob_avail_countries = [c for c in all_countries if mob_source in analysis_paths[c]]
    for cont in CONT_CMAP:
        cont_countries = [c for c in countries_by_cont[cont] if c in mob_avail_countries]
        display(Markdown(f"#### {pc.convert_continent_code_to_continent_name(cont)}"))
        for countries in split_list_into_segments(cont_countries, 12):
            display(compare_proc_mob(analysis_paths, countries, 4, mob_type))

mob_avail_countries = [c for c in all_countries if "g_mob" in analysis_paths[c]]
display(Markdown("## Weighted scaling process versus residual comparison\n\n"))
for cont in CONT_CMAP:
    display(Markdown(f"### {pc.convert_continent_code_to_continent_name(cont)}"))
    
    cont_countries = [c for c in countries_by_cont[cont] if c in mob_avail_countries]
    for countries in split_list_into_segments(cont_countries, 16):
        display(compare_proc_weighted_gmob(analysis_paths, countries, 200, 4))

# Comparisons of the non-mechanistic scaling of transmission process under each mobility configuration
This section presents comparisons of residual transmission scaling
across each mobility analysis approach for each country,
along with the 95% credible interval associated with each.
We would expect that approaches to analysis that
if applying mobility captured some part of the variation 
in transmission that would otherwise need to be 
included through residual variation,
then that mobility approach would be associated with
less dramatic changes in the residual transmission scaling.

## Results by continent

In [ ]:
procs = get_country_procs(analysis_paths, all_countries)
for cont in countries_by_cont:
    cont_name = pc.convert_continent_code_to_continent_name(cont)
    display(Markdown(f"### {cont_name}"))
    cont_countries = countries_by_cont[cont]
    for countries in split_list_into_segments(cont_countries, 9):
        display(plot_proc_comparison(procs, countries, analysis_paths))

# Comparisons of the non-mechanistic transmission scaling dispersion parameter
This section presents results based on the posterior distribution 
of the residual transmission scaling dispersion parameter.
This quantity governs the distribution in the change in residual scaling for transmission
from one value in the process series to the subsequent update.
As such, smaller values imply that smaller updates could still
result in good calibrations.
Lower values for this parameter can therefore be interpreted
as less dramatic changes needing to be applied through
the non-mechanistic component of the model.
As such, we interpreted mobility analysis approaches 
for which this posterior distribution of this parameter
was lower as being a more plausible representation of reality.
This is intended as a more formal quantification of the 
differences in variation in the non-mechanistic transmission scaling
presented in the previous section.

## Dispersion parameter distributions by analysis and country

In [ ]:
disp_posts = {c: get_param_vals_by_analysis("dispersion_proc", analysis_paths[c]) for c in all_countries}
param = "dispersion_proc"
note = (
    "Note that for the countries of Oceania, a different baseline analysis "
    "was used for to compare Google and Facebook analyses against. "
    "This was done because the conclusion of the Facebook data meant "
    "that the analysis period differed from the Google analysis."
)
param_name = yml.safe_load(open(DATA_PATH / "evidence/priors.yml", "r"))["other"]["dispersion_proc"]["short_name"]
axis_adjustments = {"URY": [-0.05, 0.35]}
for c, (cont, cont_countries) in enumerate(countries_by_cont.items()):
    if c:
        display(Latex(r"\newpage"))
    cont_name = pc.convert_continent_code_to_continent_name(cont)
    display(Markdown(f"### {cont_name}"))
    title = f"Posterior distribution for {param_name} parameter, {cont_name}"
    if cont == "OC":
        display(Markdown(note))
    for countries in split_list_into_segments(cont_countries, 16):
        param_posts = {iso3: disp_posts[iso3] for iso3 in countries}
        display(plot_kde_comparison(param_posts, axis_adjustments)) 

# Mobility exponent parameter posteriors
This section presents estimates of 
the mobility exponent parameter posteriors by country and mobility analysis type.
Values from zero to two were considered plausible 
(the prior for this parameter was uniform over domain $[0, 2]$). 
A value of zero represents no effect, 
a value of one represents a linear scaling in transmission with mobility changes,
and a value of two represents a quadratic scaling in transmission with mobility changes 
(which could be epidemiologically considered as an effect on both infector and infectee).
For all violin plots, the vertical axis represents the mobility exponent parameter distribution,
while the darkness of the coloured shading represents the median of the relative
values of the dispersion parameter posterior under the analysis with mobility included
compared against the baseline simulation.

## Mobility exponent parameter posterior distributions by country

In [ ]:
disp_posts = {c: get_param_vals_by_analysis("dispersion_proc", analysis_paths[c]) for c in all_countries}
ratio_dists = get_ratios_from_disps(disp_posts)
analyses = {k: v for k, v in ANALYSIS_NAMES.items() if "no_mob" not in k}

for mob_source, mob_name in analyses.items():

    # Median ratios
    ratio_vals = get_median_ratios(ratio_dists, mob_source)
    ratios = {get_country_short_name(k): v for k, v in ratio_vals.items()}
    
    # Mobility exponent distributions
    idatas, _ = get_idatas_for_mob_type(analysis_paths, all_countries, mob_source)
    mob_exp_df = pd.DataFrame({c: idatas[c].posterior["mob_exp"].to_series() for c in idatas})

    # Plot
    display(Markdown(f"### {mob_name}"))
    display(plot_mob_exp_violins(mob_source, mob_exp_df, ratios))

# Mobility weight posteriors
This document presents results for the weights
for each Google mobility location.
Under the Google mobility analysis approach,
multiple locations were available.
To create an overall time series of Google mobility,
we weighted each contribution according to a calibrated parameter
ranging between zero and one, which was then normalised to sum to one.
The distribution of the parameter for each location is presented as follows.

{{< pagebreak >}}

## Results by continent

In [ ]:
g_mob_countries = [c for c in all_countries if "g_mob" in analysis_paths[c]]
countries_by_cont = get_countries_by_continent(g_mob_countries)
for ct, (cont, cont_countries) in enumerate(countries_by_cont.items()):
    if ct:
        display(Latex(r"\newpage"))
    cont_name = pc.convert_continent_code_to_continent_name(cont)
    display(Markdown(f"### {cont_name}"))
    for countries in split_list_into_segments(cont_countries, 16):
        display(plot_mob_weights_by_country(analysis_paths, countries))

# Effect of waning immunity
This section presents comparisons of key model parameters with and without the inclusion of waning immunity.

\newpage

## Example dispersion parameter posterior distributions
The following plots show the posterior distribution of the dispersion to the variable process
with and without the incorporation of waning immunity into the model.

In [ ]:
waning_paths = get_analysis_paths(WANING_RERUNS + WANING_COMPARISON + WANING_TIMEOUTS, all_countries)
n_samples = 12
sample_countries = random.sample(list(analysis_paths), n_samples)
sample_analyses = [[iso3, random.sample(list(analysis_paths[iso3]), 1)[0]] for iso3 in sample_countries]

n_analyses = sum(len(p) for p in analysis_paths.values())
txt = f"These are presented for a random sample of country-analysis combinations sampled from the {n_analyses} total analyses, "
txt += f"with {n_samples} random analyses were selected for display."
Markdown(txt)

In [ ]:
plot_waning_comparison_proc_disp(waning_paths, analysis_paths, n_samples, sample_analyses)

\newpage
## Quantile-quantile posterior comparisons

The following quantile-quantile graph shows 
the proportion of runs under the analysis that included waning immunity
for which the variable process dispersion was 
greater than under the base case analysis,
for the equivalent country-analysis type combination.

In [ ]:
quant_df = get_quantmedian_df(waning_paths, analysis_paths)
prop_40_60 = round(quant_df.stack().between(0.4, 0.6).mean(), 3) * 100
txt = f"Across all country-analysis type combinations, {prop_40_60} per cent of values fell between 0.4 and 0.6."
Markdown(txt)

\vspace{1cm}

In [ ]:
plot_waning_quant_comparison(quant_df)

\newpage
The individual proportions for each country-analysis combination are listed in the following table.
\vspace{1cm}

In [ ]:
Markdown(convert_quant_df_for_display(quant_df).to_markdown())

# Identifiability analysis
The following analyses are intended to show the models ability to
recover key known parameters through its calibration algorithm.

## Analysis steps
The steps of this analysis are as follows:

1. One country was selected from the countries included in the main analysis other than those from Oceania (without replacement on iterations)
2. One mobility approach was selected from the types of mobility analysis that were run for that country
3. One accepted parameter set was randomly selected from that country-mobility type calibration
4. Only the targets for deaths and cases were identified from this analysis
5. A calibration was run for that country-mobility type combination, with all parameters fixed at the values selected in Step 3 - except for the variable process and the mobility exponent mapping parameter (if mobility was included), which were calibrated
6. The results were plotted as below
The upper panels of figures below compare the calibration fit to data for cases and deaths. The lower-left panel compares the target values for the variable process from the base case analysis (red circles) against the variable process estimates from this re-calibration analysis (black lines). The lower-right panel compares the target value for the mobility exponent mapping parameter (vertical blue bar) against the estimated posterior density for this re-calibration analysis (red area).

\newpage
## Analysis results

In [ ]:
analysis_time = "20260203_1740"
job_path = BASE_PATH / "identify_outputs" / analysis_time
countries = ls(job_path)
for iso3 in countries:
    country = pycountry.countries.lookup(iso3).name
    mob_source = ls(job_path / iso3)[0]
    display(Markdown(f"### {country}, {ANALYSIS_NAMES[mob_source]}"))
    path = job_path / iso3 / mob_source
    idata = az.from_netcdf( path/ "idata_filtered.nc")
    targets = load_targets(path)
    spaghetti = pd.read_hdf(path / "spaghetti.h5")
    updates = pd.read_hdf(path / "updates.h5")
    scalar_params = pickle.load(open(path / "scalar_params.pkl", "rb"))
    multi_params = pickle.load(open(path / "multi_params.pkl", "rb"))    
    display(plot_input_recovery(scalar_params, multi_params, idata, targets, spaghetti, updates))

{{< pagebreak >}}

# Commits used for analyses
For reproducbility, the following table gives the (short) commit SHA for each analysis.

In [ ]:
Markdown(get_analysis_commits_df(analysis_paths).to_markdown())

{{< pagebreak >}}

# References